# KnowledgeHub RAG v0.6.2 – Gold-Set Evaluation

## Objective

v0.6.2 adds a structured evaluation layer to the conversational RAG system introduced in v0.6.1.

The goal of this release is to measure whether the system retrieves the correct document evidence and produces grounded answers for both answerable and unanswerable questions. This creates a repeatable baseline for identifying retrieval failures, hallucinations, and weaknesses in the generation model before making further improvements.

## What changed in this version

- Added a gold evaluation set containing:
  - Answerable document questions
  - Expected answers or required keywords
  - Expected source pages/chunks
  - Unanswerable and out-of-document questions
- Added retrieval evaluation to check whether the correct source chunk appears in the retrieved results.
- Added answer evaluation using expected keywords.
- Added a combined RAG evaluation workflow for running the full test set.
- Preserved all v0.6.1 functionality:
  - Conversation memory
  - Follow-up question handling
  - FAISS semantic retrieval
  - BM25 keyword retrieval
  - Query expansion
  - Weighted score fusion
  - Reciprocal Rank Fusion (RRF)
  - Context expansion
  - TinyLlama answer generation

## Workflow

User Question  
↓  
Conversation Memory  
↓  
Query Expansion  
↓  
FAISS Semantic Search + BM25 Retrieval  
↓  
Weighted Fusion + Reciprocal Rank Fusion  
↓  
Context Expansion  
↓  
TinyLlama Response  
↓  
Gold-Set Evaluation

## Why this version matters

v0.6.2 turns the project from a system that can be demonstrated manually into one that can be tested systematically.

The evaluation results provide the evidence used to guide later changes, including the stronger Qwen model and confidence-based Don’t Know mode introduced in v0.6.3.

In [1]:
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q pypdf
!pip install -q transformers
!pip install -q accelerate
!pip install -q rank-bm25
!pip install -q langchain

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378.1/378.1 kB 7.4 MB/s eta 0:00:00


In [2]:
import os
import time
import faiss
import numpy as np

from pypdf import PdfReader

from sentence_transformers import SentenceTransformer

from transformers import pipeline

from rank_bm25 import BM25Okapi

In [3]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os

DATA_PATH = "/content/drive/MyDrive/KnowledgeHub_RAG/data"

pdf_files = [
    os.path.join(DATA_PATH, file)
    for file in os.listdir(DATA_PATH)
    if file.lower().endswith(".pdf")
]

print(f"Found {len(pdf_files)} PDF(s):")

for pdf in pdf_files:
    print("-", os.path.basename(pdf))

Found 1 PDF(s):
- Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf


In [5]:
def load_pdf(pdf_path):

    reader = PdfReader(pdf_path)

    pages = []

    for page_number, page in enumerate(reader.pages, start=1):

        extracted = page.extract_text()

        if extracted:

            pages.append(
                {
                    "page": page_number,
                    "text": extracted
                }
            )

    return pages


documents = []

for pdf in pdf_files:

    pages = load_pdf(pdf)

    documents.append(
        {
            "filename": os.path.basename(pdf),
            "pages": pages
        }
    )

print(f"Loaded {len(documents)} document(s).")

Loaded 1 document(s).


In [6]:
!pip install -q langchain-text-splitters

In [7]:
import re

from langchain_text_splitters import RecursiveCharacterTextSplitter


# ==========================================================
# Heading Detection
# ==========================================================

def is_heading(line):

    line = line.strip()

    if len(line) < 2:
        return False

    # 1
    # 1.2
    # 2.3.4
    if re.match(r"^\d+(\.\d+)*\s+[A-Z]", line):
        return True

    # ALL CAPS
    if line.isupper() and len(line.split()) <= 8:
        return True

    # Markdown
    if line.startswith("#"):
        return True

    # Very short title
    if len(line.split()) <= 8 and line.endswith(":"):
        return True

    return False


# ==========================================================
# Split page into sections using headings
# ==========================================================

def split_into_sections(text):

    lines = text.split("\n")

    sections = []

    current = []

    for line in lines:

        if is_heading(line):

            if current:
                sections.append("\n".join(current).strip())

            current = [line]

        else:

            current.append(line)

    if current:
        sections.append("\n".join(current).strip())

    return sections


# ==========================================================
# Recursive splitter
# ==========================================================

splitter = RecursiveCharacterTextSplitter(

    chunk_size=800,

    chunk_overlap=150,

    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)


# ==========================================================
# Final Chunking Function
# ==========================================================

def chunk_text(text):

    sections = split_into_sections(text)

    final_chunks = []

    for section in sections:

        if len(section) <= 900:

            final_chunks.append(section)

        else:

            final_chunks.extend(
                splitter.split_text(section)
            )

    return final_chunks

In [8]:
all_chunks = []

chunk_id = 0

for document in documents:

    for page in document["pages"]:

        sections = split_into_sections(page["text"])

        for section in sections:

            # -----------------------------
            # Extract heading
            # -----------------------------
            lines = section.split("\n")

            heading = ""

            if lines and is_heading(lines[0]):
                heading = lines[0].strip()

            # -----------------------------
            # Split section if needed
            # -----------------------------
            if len(section) <= 900:

                section_chunks = [section]

            else:

                section_chunks = splitter.split_text(section)

            # -----------------------------
            # Save chunks
            # -----------------------------
            for chunk in section_chunks:

                all_chunks.append(
                    {
                        "chunk_id": chunk_id,
                        "document": document["filename"],
                        "page": page["page"],
                        "section": heading,
                        "text": chunk
                    }
                )

                chunk_id += 1

print(f"Created {len(all_chunks)} chunks.")

Created 112 chunks.


In [9]:
print("=" * 80)

print("FIRST 10 CHUNKS")

print("=" * 80)

for chunk in all_chunks[:10]:

    print(f"\nChunk ID : {chunk['chunk_id']}")

    print(f"Page     : {chunk['page']}")


    print("-" * 80)

    print(chunk["text"][:400])

    print()

FIRST 10 CHUNKS

Chunk ID : 0
Page     : 1
--------------------------------------------------------------------------------
Machine Learning Classification of Binary Neutron
Star Remnants Using Gravitational Wave Data
Surendaranath Kanniyappan
Dr. Michalis Agathos
Abstract
Binary neutron star (BNS) mergers are among the most energetic cosmic events,
producing gravitational waves (GWs), electromagnetic (EM) counterparts, and potentially
neutrinos. These mergers provide an unparalleled opportunity to study supranuclear m


Chunk ID : 1
Page     : 1
--------------------------------------------------------------------------------
hypermassive neutron star (short- or long-lived HMNS), or remains stable.
Direct detection of postmerger GW signals remains challenging due to their high
frequency nature (≳ 1 kHz) and the sensitivity limits of current interferometers. Therefore,
predicting remnant outcomes from inspiral parameters—total mass Mtot, mass ratio q, tidal
deformability ˜Λ, and effecti

In [10]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [11]:
texts = [chunk["text"] for chunk in all_chunks]

embeddings = embedding_model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

print(embeddings.shape)

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

(112, 384)


In [12]:
import faiss

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(
    embeddings.astype("float32")
)

print(index.ntotal)

112


In [13]:

tokenized_corpus = [
    chunk["text"].lower().split()
    for chunk in all_chunks
]

bm25 = BM25Okapi(tokenized_corpus)

print(" BM25 Index Built")

 BM25 Index Built


In [14]:
QUERY_EXPANSION = {

    "algorithm": [
        "model",
        "classifier",
        "GBDT",
        "Gradient Boosted Decision Tree"
    ],

    "gbdt": [
        "Gradient Boosted Decision Tree",
        "gradient boosting"
    ],

    "classifier": [
        "classification model",
        "machine learning model"
    ],

    "accuracy": [
        "performance",
        "evaluation",
        "MCC"
    ],

    "dataset": [
        "training data",
        "simulation dataset"
    ],

    "method": [
        "approach",
        "framework"
    ]
}


def expand_query(query):

    expanded = query

    query_lower = query.lower()

    for key, values in QUERY_EXPANSION.items():

        if key in query_lower:

            expanded += " " + " ".join(values)

    return expanded

In [15]:
def reciprocal_rank_fusion(semantic_results, bm25_results, k=60):
    """
    Reciprocal Rank Fusion (RRF)

    Score(doc) = Σ 1 / (k + rank)

    Larger k -> smoother scores (60 is the standard value).
    """

    fused = {}

    # Semantic ranking
    for rank, item in enumerate(semantic_results, start=1):
        cid = item["chunk_id"]

        if cid not in fused:
            fused[cid] = item.copy()
            fused[cid]["rrf_score"] = 0.0

        fused[cid]["rrf_score"] += 1.0 / (k + rank)

    # BM25 ranking
    for rank, item in enumerate(bm25_results, start=1):
        cid = item["chunk_id"]

        if cid not in fused:
            fused[cid] = item.copy()
            fused[cid]["rrf_score"] = 0.0

        fused[cid]["rrf_score"] += 1.0 / (k + rank)

    return sorted(
        fused.values(),
        key=lambda x: x["rrf_score"],
        reverse=True
    )

In [16]:
def retrieve(query, top_k=5):

    # =====================================================
    # Encode Query
    # =====================================================
    expanded_query = expand_query(query)

    query_embedding = embedding_model.encode(
        [expanded_query],
        normalize_embeddings=True
    ).astype("float32")

    # =====================================================
    # FAISS Search
    # =====================================================

    semantic_scores, semantic_indices = index.search(
        query_embedding,
        top_k * 8
    )

    semantic_results = []

    for rank, (score, idx) in enumerate(
        zip(semantic_scores[0], semantic_indices[0]),
        start=1
    ):

        semantic_results.append({

            "chunk_id": idx,
            "rank": rank,
            "semantic_score": float(score)

        })

    # =====================================================
    # BM25 Search
    # =====================================================

    tokenized_query = expanded_query.lower().split()

    bm25_scores = bm25.get_scores(tokenized_query)

    bm25_ranked = sorted(

        enumerate(bm25_scores),

        key=lambda x: x[1],

        reverse=True

    )[:top_k * 8]

    bm25_results = []

    for rank, (idx, score) in enumerate(
        bm25_ranked,
        start=1
    ):

        bm25_results.append({

            "chunk_id": idx,
            "rank": rank,
            "bm25_score": float(score)

        })

    # =====================================================
    # Reciprocal Rank Fusion (RRF)
    # =====================================================

    k = 60

    rrf_scores = {}

    semantic_lookup = {}
    bm25_lookup = {}

    for item in semantic_results:

        cid = item["chunk_id"]

        semantic_lookup[cid] = item["semantic_score"]

        rrf_scores[cid] = rrf_scores.get(cid, 0) + 1 / (k + item["rank"])

    for item in bm25_results:

        cid = item["chunk_id"]

        bm25_lookup[cid] = item["bm25_score"]

        rrf_scores[cid] = rrf_scores.get(cid, 0) + 1 / (k + item["rank"])

    rrf_results = []

    for chunk_id, rrf_score in rrf_scores.items():

        chunk = all_chunks[chunk_id]

        rrf_results.append({

            "chunk_id": chunk_id,

            "page": chunk["page"],

            "document": chunk["document"],

            "text": chunk["text"],

            "semantic_score": semantic_lookup.get(chunk_id, 0.0),

            "bm25_score": bm25_lookup.get(chunk_id, 0.0),

            "rrf_score": rrf_score

        })

    # =====================================================
    # Intelligent Tie Breaking
    # =====================================================

    for item in rrf_results:

        item["final_score"] = (

            item["rrf_score"]

            + 0.001 * item["semantic_score"]

            + 0.001 * item["bm25_score"]

        )

    rrf_results = sorted(

        rrf_results,

        key=lambda x: x["final_score"],

        reverse=True

    )

    # =====================================================
    # Prevent Adjacent Matched Chunks
    # =====================================================

    selected = []

    for item in rrf_results:

        current = item["chunk_id"]

        if any(abs(current - x["chunk_id"]) <= 1 for x in selected):
            continue

        selected.append(item)

        if len(selected) == top_k:
            break

    # =====================================================
    # Context Expansion
    # =====================================================

    expanded = []

    visited = set()

    for rank, item in enumerate(selected, start=1):

        current = item["chunk_id"]

        for neighbour in [current - 1, current, current + 1]:

            if neighbour < 0:
                continue

            if neighbour >= len(all_chunks):
                continue

            if neighbour in visited:
                continue

            if all_chunks[neighbour]["document"] != item["document"]:
                continue

            visited.add(neighbour)

            chunk = all_chunks[neighbour]

            expanded.append({

                "chunk_id": chunk["chunk_id"],

                "page": chunk["page"],

                "document": chunk["document"],

                "text": chunk["text"],

                "retrieval_rank": rank,

                "context_neighbor": neighbour != current,

                "semantic_score": item["semantic_score"] if neighbour == current else None,

                "bm25_score": item["bm25_score"] if neighbour == current else None,

                "rrf_score": item["rrf_score"] if neighbour == current else None

            })

    return expanded

In [17]:
def debug_retrieval(query, top_k=5):

    results = retrieve(query, top_k)

    print("=" * 90)
    print(f"QUERY : {query}")
    print("=" * 90)

    current_rank = None

    for chunk in results:

        if chunk["retrieval_rank"] != current_rank:

            current_rank = chunk["retrieval_rank"]

            print()
            print("=" * 90)
            print(f"RETRIEVAL RANK {current_rank}")
            print("=" * 90)

        print()

        if chunk["context_neighbor"]:

            print("Context Chunk")

        else:

            print("Matched Chunk")

            print(f"Semantic Score : {chunk['semantic_score']:.4f}")
            print(f"BM25 Score     : {chunk['bm25_score']:.4f}")
            if "rrf_score" in chunk:
              print(f"RRF Score      : {chunk['rrf_score']:.5f}")

        print(f"Document : {chunk['document']}")
        print(f"Page     : {chunk['page']}")

        if chunk.get("section"):
          print(f"Section  : {chunk['section']}")

        print(f"Chunk ID : {chunk['chunk_id']}")

        print("-" * 90)

        print(chunk["text"][:700])

        print()

In [18]:
debug_retrieval("What algorithm was used for classification?")

debug_retrieval("What is GBDT?")

debug_retrieval("Gradient Boosted Decision Tree")

QUERY : What algorithm was used for classification?

RETRIEVAL RANK 1

Context Chunk
Document : Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf
Page     : 8
Chunk ID : 33
------------------------------------------------------------------------------------------
difficulty-aware splitting strategy described in [11] is adopted:
1. Multiple candidate splits were generated at random while maintaining stratification.
2. Trial models were trained on each candidate split to identify samples that were frequently
misclassified across trials.
3. Misclassification frequency was used to bin the data into “easy” and “hard” subsets.
4. Balanced sampling ensured that each bin contributed proportionally to both training and
validation sets.
This procedure yields validation sets that are more representative of the full complexity of the
dataset, leading to more realistic and stable performance estimates.


Matched Chunk
Semantic Score : 0.6347
BM25 Score     : 13.7764
RRF Score      : 0.03

In [19]:
query = "What algorithm was used for classification?"

results = retrieve(query)

# Keep only the best 2 chunks
context = "\n\n".join(
    [r["text"][:600] for r in results[:2]]
)

print("=" * 80)
print(context)

difficulty-aware splitting strategy described in [11] is adopted:
1. Multiple candidate splits were generated at random while maintaining stratification.
2. Trial models were trained on each candidate split to identify samples that were frequently
misclassified across trials.
3. Misclassification frequency was used to bin the data into “easy” and “hard” subsets.
4. Balanced sampling ensured that each bin contributed proportionally to both training and
validation sets.
This procedure yields validation sets that are more representative of the full complexity of the
dataset, leading to more reali

3.5 Gradient Boosted Decision Tree (GBDT) Training
All three classifiers were trained using the gradient boosting framework for decision trees [27]
as implemented in scikit-learn [28]. This method builds an ensemble of shallow decision
trees (weak learners) sequentially, with each new tree correcting the residual errors of the
previous ensemble. The outputs are combined, weighted by a learning r

In [20]:
query = "What algorithm was used for classification?"

results = retrieve(query)

print(f"Retrieved {len(results)} chunks.")

Retrieved 15 chunks.


In [21]:
TOP_CONTEXT = 3

context = "\n\n".join(
    chunk["text"]
    for chunk in results[:TOP_CONTEXT]
)

print("=" * 80)
print(context)

difficulty-aware splitting strategy described in [11] is adopted:
1. Multiple candidate splits were generated at random while maintaining stratification.
2. Trial models were trained on each candidate split to identify samples that were frequently
misclassified across trials.
3. Misclassification frequency was used to bin the data into “easy” and “hard” subsets.
4. Balanced sampling ensured that each bin contributed proportionally to both training and
validation sets.
This procedure yields validation sets that are more representative of the full complexity of the
dataset, leading to more realistic and stable performance estimates.

3.5 Gradient Boosted Decision Tree (GBDT) Training
All three classifiers were trained using the gradient boosting framework for decision trees [27]
as implemented in scikit-learn [28]. This method builds an ensemble of shallow decision
trees (weak learners) sequentially, with each new tree correcting the residual errors of the
previous ensemble. The outputs 

In [22]:
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM

import torch

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading model...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Prevent generation warnings
model.generation_config.max_length = None
model.generation_config.pad_token_id = tokenizer.eos_token_id

print("TinyLlama Loaded")

Loading tokenizer...


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

TinyLlama Loaded


In [23]:
def build_prompt(question, context):

    history = get_chat_history()

    return f"""
You are an AI assistant answering questions from a research report.

Rules:

- Answer ONLY using the Context.
- Do NOT use outside knowledge.
- Do NOT invent facts or names.
- If the answer is not present in the context, reply:

I could not find that information in the provided document.

- Keep answers concise.
- Maximum 3 sentences.

Previous Conversation:
{history}

Context:
{context}

Question:
{question}

Answer:
"""

In [24]:
# ----------------------------
# Conversation Memory
# ----------------------------

conversation_history = []


def get_chat_history(max_turns=3):
    """
    Returns the previous conversation history
    as formatted text.
    """

    if len(conversation_history) == 0:
        return ""

    history = ""

    for q, a in conversation_history[-max_turns:]:

        history += f"User: {q}\n"
        history += f"Assistant: {a}\n\n"

    return history

In [25]:
def generate_answer(question, context):

    prompt = build_prompt(question, context)

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        do_sample=False,
        repetition_penalty=1.15,
        max_new_tokens=120,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )

    full_output = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    answer = full_output[len(prompt):].strip()

    # Prevent the model from continuing the prompt
    stop_words = [
        "\nQuestion:",
        "\nContext:",
        "\nUser:",
        "\nAssistant:"
    ]

    for stop in stop_words:
        if stop in answer:
            answer = answer.split(stop)[0].strip()

    return answer

In [26]:
query = "What algorithm was used for classification?"

results = retrieve(query)

context = "\n\n".join(
    chunk["text"]
    for chunk in results[:3]
)

answer = generate_answer(query, context)

print("=" * 80)
print("QUESTION")
print(query)

print("\n" + "=" * 80)
print("ANSWER")
print(answer)

QUESTION
What algorithm was used for classification?

ANSWER
Gradient Boosted Decision Trees (GBDT).


In [27]:
while True:
    print("\n" + "=" * 80)

    question = input("\nAsk a question (type 'exit' to quit): ")

    if question.lower() == "exit":
        print("\nGoodbye!")
        break

    results = retrieve(question)

    context = "\n\n".join(
        chunk["text"]
        for chunk in results[:4]
    )

    answer = generate_answer(
    question,
    context
    )

    conversation_history.append((question, answer))

    print("\n" + "=" * 80)
    print(answer)




Ask a question (type 'exit' to quit): What is the accuracy?

The accuracy is 0.956.


Ask a question (type 'exit' to quit): exit

Goodbye!


## Known Limitations (v0.6.1)

This version introduces conversational memory, enabling multi-turn question answering over uploaded documents.

Although retrieval has been significantly improved through hybrid search, query expansion, weighted score fusion, Reciprocal Rank Fusion (RRF), and context expansion, answer generation is still performed by the lightweight TinyLlama (1.1B) language model.

Current limitations include:

- The model may occasionally generate unsupported details when summarising long contexts.
- Out-of-document questions may be answered using the model's pretrained knowledge instead of responding that the information is unavailable.
- Response quality remains dependent on the relevance of the retrieved document chunks.

These limitations will be addressed in future releases through retrieval confidence thresholds, a "don't know" response mechanism, automated evaluation, and stronger open-source language models.

In [28]:
query = "Which sport is most played?"

results = retrieve(query)

context = "\n\n".join(
    chunk["text"]
    for chunk in results[:3]
)

answer = generate_answer(query, context)

print("=" * 80)
print("QUESTION")
print(query)

print("\n" + "=" * 80)
print("ANSWER")
print(answer)

QUESTION
Which sport is most played?

ANSWER
Football


###Gold Evaluation Set

In [29]:
questions = [
    "What algorithm was used for classification?",
    "How was the dataset split?",
    "Why was GBDT chosen?",
    "Which inspiral parameters were used?",
    "What does Classifier A predict?",
    "Who is the author of the report?",
    "What is SHAP used for?",
    "Which real gravitational-wave events were analysed?",
    "What optimizer was used to train the neural network?",
    "What GPU was used for training?"
]

In [30]:
for q in questions:
    debug_retrieval(q)

QUERY : What algorithm was used for classification?

RETRIEVAL RANK 1

Context Chunk
Document : Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf
Page     : 8
Chunk ID : 33
------------------------------------------------------------------------------------------
difficulty-aware splitting strategy described in [11] is adopted:
1. Multiple candidate splits were generated at random while maintaining stratification.
2. Trial models were trained on each candidate split to identify samples that were frequently
misclassified across trials.
3. Misclassification frequency was used to bin the data into “easy” and “hard” subsets.
4. Balanced sampling ensured that each bin contributed proportionally to both training and
validation sets.
This procedure yields validation sets that are more representative of the full complexity of the
dataset, leading to more realistic and stable performance estimates.


Matched Chunk
Semantic Score : 0.6347
BM25 Score     : 13.7764
RRF Score      : 0.03

In [31]:
import json

In [32]:
with open("gold_eval.json") as f:
    gold = json.load(f)

In [33]:
def evaluate_retrieval(retrieved_chunks, sample):

    if not sample["answerable"]:
        return True

    retrieved_pages = [
        chunk["page"]
        for chunk in retrieved_chunks[:3]
    ]

    retrieved_chunk_ids = [
        chunk["chunk_id"]
        for chunk in retrieved_chunks[:3]
    ]

    passed = (
        sample["gold_chunk"] in retrieved_chunk_ids
        or
        sample["gold_page"] in retrieved_pages
    )

    print("Retrieval :", "PASS" if passed else "FAIL")
    print("Expected Page :", sample["gold_page"])
    print("Retrieved Pages :", retrieved_pages)

    return passed

In [34]:
def evaluate_generation(sample, retrieved_chunks):

    context = "\n\n".join(
        chunk["text"]
        for chunk in retrieved_chunks[:3]
    )

    answer = generate_answer(
        sample["question"],
        context
    )

    print("\nGenerated Answer\n")
    print(answer)

    return answer

In [35]:
def evaluate_answer(answer, sample):

    if not sample["answerable"]:

        passed = (
            "could not find" in answer.lower()
            or
            "not found" in answer.lower()
        )

    else:

        hits = sum(
            keyword.lower() in answer.lower()
            for keyword in sample["keywords"]
        )

        passed = (
            hits / len(sample["keywords"])
        ) >= 0.65

    print("\nExpected Answer\n")
    print(sample["expected_answer"])

    print("\nGeneration :", "PASS" if passed else "FAIL")

    return passed

In [36]:
def evaluate_rag():

    retrieval_correct = 0
    generation_correct = 0

    total = len(gold)

    for sample in gold:

        print("=" * 80)
        print("QUESTION")
        print(sample["question"])
        print("=" * 80)

        retrieved_chunks = retrieve(sample["question"])

        retrieval_pass = evaluate_retrieval(
            retrieved_chunks,
            sample
        )

        if retrieval_pass:
            retrieval_correct += 1

        answer = evaluate_generation(
            sample,
            retrieved_chunks
        )

        generation_pass = evaluate_answer(
            answer,
            sample
        )

        if generation_pass:
            generation_correct += 1

    print("\n" + "=" * 80)

    print(f"Retrieval Accuracy : {retrieval_correct}/{total} ({100*retrieval_correct/total:.1f}%)")
    print(f"Generation Accuracy: {generation_correct}/{total} ({100*generation_correct/total:.1f}%)")

In [37]:
def evaluate_subset(indices):

    retrieval_correct = 0
    generation_correct = 0

    for idx in indices:

        sample = gold[idx]

        print("=" * 80)
        print("QUESTION")
        print(sample["question"])
        print("=" * 80)

        retrieved_chunks = retrieve(sample["question"])

        retrieval_pass = evaluate_retrieval(
            retrieved_chunks,
            sample
        )

        if retrieval_pass:
            retrieval_correct += 1

        answer = evaluate_generation(
            sample,
            retrieved_chunks
        )

        generation_pass = evaluate_answer(
            answer,
            sample
        )

        if generation_pass:
            generation_correct += 1

    print("\n" + "=" * 80)

    print(
        f"Retrieval Accuracy : {retrieval_correct}/{len(indices)} "
        f"({100*retrieval_correct/len(indices):.1f}%)"
    )

    print(
        f"Generation Accuracy: {generation_correct}/{len(indices)} "
        f"({100*generation_correct/len(indices):.1f}%)"
    )

In [38]:

evaluate_subset([6])

QUESTION
What is SHAP used for?
Retrieval : PASS
Expected Page : 9
Retrieved Pages : [8, 9, 9]

Generated Answer

SHAP is used to understand how different features contribute to the predicted outcome. It provides a way to visualize the relationship between input variables and output variable.

Previous Conversation:

Expected Answer

To interpret model predictions and rank feature importance.

Generation : PASS

Retrieval Accuracy : 1/1 (100.0%)
Generation Accuracy: 1/1 (100.0%)


In [39]:
evaluate_rag()

QUESTION
What algorithm was used for classification?
Retrieval : PASS
Expected Page : 8
Retrieved Pages : [8, 8, 8]

Generated Answer

Gradient Boosted Decision Trees (GBDT).

Expected Answer

Gradient Boosted Decision Trees (GBDT).

Generation : PASS
QUESTION
How was the dataset split?
Retrieval : PASS
Expected Page : 8
Retrieved Pages : [7, 8, 8]

Generated Answer

The dataset was divided into three parts: training, validation, and test. The training set contained 90% of the data, while the validation set contained 10%. The test set contained the remaining 10%.

Previous Conversation:

Expected Answer

90% training and 10% validation using stratified, difficulty-aware splitting.

Generation : PASS
QUESTION
Why was GBDT chosen?
Retrieval : PASS
Expected Page : 8
Retrieved Pages : [8, 8, 8]

Generated Answer

Because it captures complex, non-linear decision boundaries, which can be difficult to model using other methods. It also performs well on moderate sized datasets without requirin